---
title: "Data Structures and Algorithms: Computational Complexity, Hardness, and Approximation"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    # code-fold: true
jupyter: python
---


[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Computational Complexity, Hardness, and Approximation** {#computational-complexity-hardness-and-approximation}

Algorithm analysis usually begins after a problem has been chosen: given an algorithm, how much time and memory does it use? **Computational complexity theory** asks a prior question: what resources are inherently required by the problem itself, across all possible algorithms under a stated model of computation?

This shift matters when straightforward search has exponentially many candidates and years of research have not produced a polynomial-time exact algorithm. Complexity theory provides a disciplined vocabulary for separating four claims that are often confused:

- a particular implementation is slow;
- no polynomial-time algorithm is currently known;
- the problem has been proved NP-hard under a specified reduction;
- the required output or guarantee can be relaxed to obtain a useful algorithm.

The chapter uses **Vertex Cover** as a running example. Given an undirected graph $G=(V,E)$, a vertex cover is a set $C\subseteq V$ such that every edge has at least one endpoint in $C$. The optimization problem asks for the smallest cover. Its decision version asks whether a cover of size at most $k$ exists. A proposed cover is easy to verify, yet finding a minimum cover is NP-hard in general.

Hardness is not a command to stop. It changes the engineering question from "which exact polynomial algorithm did I miss?" to "which assumptions, parameters, guarantees, or resources can I exploit?" Exact exponential methods, approximation algorithms, randomized algorithms, parameterized algorithms, and heuristics all answer different versions of that question.

::: {.callout-important}
NP-hard does not mean "impossible" or "always slow." It is a worst-case statement about a family of inputs. Small instances, structured special cases, or small parameters may still be solved exactly and quickly.
:::


### **Decision Problems and Optimization Problems** {#decision-and-optimization-problems}

A **decision problem** returns one bit: YES or NO. Complexity classes such as P and NP are formally defined for decision problems because a yes/no interface makes reductions and verification precise. A **search problem** asks for a concrete witness, while an **optimization problem** asks for the best objective value or best witness.

For Vertex Cover, the three formulations are:

- **Decision:** does $G$ contain a vertex cover $C$ with $|C|\le k$?
- **Search:** return such a set $C$, if one exists.
- **Optimization:** return a cover with minimum cardinality $OPT(G)$.

These formulations are distinct interfaces, but they are often polynomially related. The decision predicate is monotone in $k$: once a cover of size $k$ exists, one also exists for every larger bound. Therefore a decision oracle can locate the optimum using binary search over $0,1,\ldots,|V|$.

Recovering a witness requires **self-reduction**. After learning the optimum $k$, tentatively remove or force a candidate choice and query whether an optimum solution remains possible. Repeating the decision query reconstructs a concrete solution. This relationship must be proved for the particular problem; it should not be assumed merely because the names sound similar.

~~~text
OPTIMUM-VERTEX-COVER-USING-DECISION(G)
    low <- 0, high <- number of vertices
    while low < high
        middle <- floor((low + high) / 2)
        if HAS-COVER(G, middle): high <- middle
        else: low <- middle + 1
    return low

RECOVER-COVER(G, optimum k)
    repeatedly test whether an optimum cover can avoid each vertex
    keep a vertex only when excluding it makes the decision answer NO
    return the forced vertices
~~~

![Decision, optimization, and search ask for different outputs but can often be connected by threshold queries and self-reduction.](assets/decision-optimization-search.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: exact Vertex Cover decision and optimization</summary>

~~~python
from itertools import combinations


def is_vertex_cover(
    edges: list[tuple[str, str]], candidate: set[str]
) -> bool:
    return all(u in candidate or v in candidate for u, v in edges)


def has_vertex_cover(
    vertices: list[str], edges: list[tuple[str, str]], limit: int
) -> bool:
    """Decision formulation: return only whether a small cover exists."""
    for size in range(min(limit, len(vertices)) + 1):
        for chosen in combinations(vertices, size):
            if is_vertex_cover(edges, set(chosen)):
                return True
    return False


def minimum_vertex_cover(
    vertices: list[str], edges: list[tuple[str, str]]
) -> set[str]:
    """Optimization formulation: return an optimal witness."""
    for size in range(len(vertices) + 1):
        for chosen in combinations(vertices, size):
            candidate = set(chosen)
            if is_vertex_cover(edges, candidate):
                return candidate
    raise AssertionError("all vertices always form a cover")


vertices = ["a", "b", "c", "d"]
edges = [("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")]
cover = minimum_vertex_cover(vertices, edges)
assert len(cover) == 2
assert has_vertex_cover(vertices, edges, 1) is False
assert has_vertex_cover(vertices, edges, 2) is True
print(cover)
~~~
</details>

The exhaustive implementation may inspect $2^{|V|}$ subsets and check $|E|$ edges for each, giving $O(2^{|V|}|E|)$ worst-case time and $O(|V|)$ candidate space. The point is not that optimization is always harder than decision; it is that the requested output must be fixed before complexity is discussed.

**Practice:** [LeetCode 698 - Partition to K Equal Sum Subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/) is naturally phrased as a decision problem and illustrates how a yes/no formulation can still require substantial combinatorial search.


### **Tractability and Computational Resources** {#tractability-and-computational-resources}

A problem is commonly called **tractable** when it has an algorithm whose worst-case resource use is polynomial in the encoded input length. If the input occupies $n$ bits or symbols, a polynomial bound has the form $O(n^c)$ for a constant $c$ independent of the instance.

This definition depends on encoding. An integer $W$ written in binary occupies $\Theta(\log W)$ bits. An $O(nW)$ dynamic program is polynomial in the numeric value $W$ but may be exponential in the number of bits needed to represent $W$. Such an algorithm is **pseudopolynomial**, not polynomial in the standard bit model.

Important resource measures include:

- **time**, usually the number of elementary computation steps;
- **space**, the maximum additional memory used;
- **random bits**, oracle queries, communication, or parallel depth in specialized models;
- **parameter dependence**, such as $f(k)n^c$ when a structural parameter $k$ is separated from total input size.

Polynomial time is a theoretical robustness criterion, not a promise of practical speed. An $O(n^{100})$ algorithm is polynomial but unusable, while an exponential algorithm may be excellent for $n\le 40$. Complexity classes describe scaling and worst-case boundaries; implementation constants, input distributions, preprocessing, and structure still matter.

~~~text
SUBSET-SUM-DP(values, target)
    reachable[0] <- true
    for each positive value x
        for total from target down to x
            reachable[total] <- reachable[total] OR reachable[total-x]
    return reachable[target]

Time: O(n * target), which is pseudopolynomial when target is binary encoded.
~~~

![Polynomial, pseudopolynomial, and exponential resource growth must be interpreted relative to encoded input length.](assets/complexity-resource-growth.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: pseudopolynomial Subset Sum</summary>

~~~python
def subset_sum(values: list[int], target: int) -> bool:
    if target < 0 or any(value < 0 for value in values):
        raise ValueError("this implementation expects non-negative integers")

    reachable = [False] * (target + 1)
    reachable[0] = True

    for value in values:
        # Descending order prevents one input value from being reused.
        for total in range(target, value - 1, -1):
            reachable[total] = reachable[total] or reachable[total - value]

    return reachable[target]


assert subset_sum([3, 7, 11, 15], 18) is True
assert subset_sum([3, 7, 11, 15], 10) is True
assert subset_sum([6, 10], 7) is False
print(subset_sum([3, 7, 11, 15], 18))
~~~
</details>

The table has <code>target + 1</code> states and each of $n$ values may update all of them, so time is $O(n\cdot target)$ and auxiliary space is $O(target)$. If <code>target</code> is exponentially larger than its binary encoding length, these bounds are not polynomial in the input bits.

**Practice:** [LeetCode 416 - Partition Equal Subset Sum](https://leetcode.com/problems/partition-equal-subset-sum/) is a direct application of pseudopolynomial Subset Sum dynamic programming.


### **P, NP, and Efficient Verification** {#p-np-and-efficient-verification}

Formally, a decision problem is represented as a **language**: the set of encoded instances whose correct answer is YES.

**P** contains decision problems solvable by a deterministic polynomial-time algorithm. **NP** contains decision problems whose YES instances have a polynomial-length **certificate** that a deterministic polynomial-time verifier can check.

Equivalently, a language $L$ is in NP when there is a polynomial-time verifier $V$ and polynomial $p$ such that

$$
x\in L
\iff
\text{there exists a certificate }y,
\ |y|\le p(|x|),
\text{ with }V(x,y)=1.
$$

Here $x$ is the encoded instance, $y$ is the witness, $|x|$ and $|y|$ are encoding lengths, and $V(x,y)=1$ means the verifier accepts. The existential phrase is essential: NO instances need not have a similarly short certificate under the definition of NP.

For Vertex Cover, the certificate is the proposed set $C$. The verifier checks $|C|\le k$ and scans every edge to ensure at least one endpoint lies in $C$. Verification is polynomial even though discovering such a set may require exponential search.

Every problem in P is also in NP because a polynomial solver can ignore the certificate and compute the answer directly. Thus

$$
P\subseteq NP.
$$

Whether $P=NP$ remains open. The name NP means **nondeterministic polynomial time**, not "non-polynomial." No unconditional proof currently permits replacing "NP-complete" with "has no polynomial algorithm."

~~~text
VERIFY-VERTEX-COVER(G=(V,E), k, certificate C)
    if C contains an unknown vertex or |C| > k: reject
    for each edge (u,v) in E
        if u not in C and v not in C: reject
    accept
~~~

![A short Vertex Cover certificate is checked edge by edge in polynomial time.](assets/certificate-verification.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: polynomial Vertex Cover verifier</summary>

~~~python
def verify_vertex_cover(
    vertices: set[str],
    edges: list[tuple[str, str]],
    limit: int,
    certificate: set[str],
) -> bool:
    # The witness must use known vertices and respect the requested bound.
    if not certificate <= vertices or len(certificate) > limit:
        return False

    # Each edge is checked once.
    return all(
        u in certificate or v in certificate
        for u, v in edges
    )


vertices = {"a", "b", "c", "d"}
edges = [("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")]
assert verify_vertex_cover(vertices, edges, 2, {"a", "c"}) is True
assert verify_vertex_cover(vertices, edges, 1, {"a"}) is False
assert verify_vertex_cover(vertices, edges, 2, {"a", "x"}) is False
print(verify_vertex_cover(vertices, edges, 2, {"a", "c"}))
~~~
</details>

With hash-set membership, verification takes expected $O(|V|+|E|)$ time including witness validation and $O(|C|)$ certificate storage. This fast verifier proves membership in NP; it does not prove NP-hardness.

**Practice:** [LeetCode 36 - Valid Sudoku](https://leetcode.com/problems/valid-sudoku/) emphasizes the verification side: checking whether a filled partial configuration violates constraints is much easier than constructing a complete valid board.


### **Polynomial-Time Reductions** {#polynomial-time-reductions}

A **polynomial-time many-one reduction** from decision problem $A$ to decision problem $B$, written

$$
A\le_p B,
$$

is a polynomial-time computable transformation $f$ satisfying

$$
x\in A \iff f(x)\in B.
$$

The transformation converts one instance, not merely one known solution. The if-and-only-if condition preserves both YES and NO instances. If a polynomial solver for $B$ existed, then transforming $x$, solving $f(x)$, and returning the same answer would solve $A$ in polynomial time. Therefore $B$ is at least as hard as $A$ under this reduction.

The direction is the most common source of mistakes. To prove a new target problem $B$ is hard, reduce a **known hard source $A$ to $B$**. Showing $B\le_p A$ only demonstrates that $B$ can use a solver for $A$.

Reductions are transitive: if $A\le_p B$ and $B\le_p C$, composing the two polynomial transformations gives $A\le_p C$. This is why one initial NP-complete problem can seed a large network of hardness results.

A simple graph reduction connects Vertex Cover and Independent Set. A set $C$ is a vertex cover exactly when its complement $V\setminus C$ is independent. Therefore

$$
(G,k)\in VertexCover
\iff
(G,|V|-k)\in IndependentSet.
$$

~~~text
VERTEX-COVER-TO-INDEPENDENT-SET(G=(V,E), k)
    return the same graph G and target size |V| - k

The graph is unchanged; only the parameter is complemented.
~~~

![The correct reduction direction transforms an instance of the known source problem into the target problem before invoking a hypothetical target solver.](assets/polynomial-reduction-direction.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: Vertex Cover to Independent Set reduction</summary>

~~~python
def vertex_cover_to_independent_set(
    vertices: set[str],
    edges: list[tuple[str, str]],
    cover_limit: int,
) -> tuple[set[str], list[tuple[str, str]], int]:
    return vertices.copy(), edges.copy(), len(vertices) - cover_limit


def is_independent_set(
    edges: list[tuple[str, str]], candidate: set[str]
) -> bool:
    return all(not (u in candidate and v in candidate) for u, v in edges)


vertices = {"a", "b", "c", "d"}
edges = [("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")]
cover = {"a", "c"}

reduced_vertices, reduced_edges, independent_target = (
    vertex_cover_to_independent_set(vertices, edges, len(cover))
)
independent = reduced_vertices - cover
assert independent_target == 2
assert is_independent_set(reduced_edges, independent)
print(independent, independent_target)
~~~
</details>

This reduction copies or references the graph and computes one integer, so it takes $O(|V|+|E|)$ time under an explicit-copy representation. The correctness comes from the complement argument, not from testing examples with code.

**Practice:** [LeetCode 1349 - Maximum Students Taking Exam](https://leetcode.com/problems/maximum-students-taking-exam/) can be modeled as an independent-set problem on a conflict graph, providing practice in translating a domain story into a known graph formulation.


### **NP-Hardness and NP-Completeness** {#np-hardness-and-np-completeness}

A problem $H$ is **NP-hard** when every problem $L\in NP$ reduces to it in polynomial time:

$$
\forall L\in NP,\quad L\le_p H.
$$

An NP-hard problem need not itself belong to NP. It may be an optimization problem rather than a decision problem, or even a problem with no polynomially verifiable certificate. **NP-complete** problems are exactly those that are both in NP and NP-hard:

$$
NPC = NP\cap NP\text{-hard}.
$$

To prove a decision problem $B$ is NP-complete, two independent obligations are required:

1. **Membership:** describe a polynomial-size certificate and polynomial-time verifier, proving $B\in NP$.
2. **Hardness:** choose a known NP-complete problem $A$ and prove $A\le_p B$.

The hardness reduction transfers all NP problems through $A$. Membership ensures the target remains within the class of efficiently verifiable decision problems. Proving only the second item establishes NP-hardness, not NP-completeness.

If any NP-complete problem has a polynomial-time algorithm, then every NP problem does and $P=NP$. Conversely, under the common but unproved assumption $P\ne NP$, no NP-complete problem lies in P.

~~~text
PROVE-NP-COMPLETE(target B)
    Membership:
        define a polynomial-length certificate for YES instances
        give a polynomial-time verifier

    Hardness:
        choose known NP-complete source A
        construct f(x) in polynomial time
        prove x is YES for A iff f(x) is YES for B
~~~

![Relationships among P, NP, NP-complete, and NP-hard problems under both possible answers to the P versus NP question.](assets/complexity-classes-p-np.svg){fig-align="center" width="82%"}

*Visual source: [Behnam Esfahbod, P/NP/NP-complete/NP-hard Euler diagram](https://commons.wikimedia.org/wiki/File:P_np_np-complete_np-hard.svg), licensed under CC BY-SA 3.0/GFDL. The unchanged SVG is stored locally.*

<details>
<summary>Python implementation: SAT verification versus exhaustive search</summary>

~~~python
from itertools import product


def verify_cnf(
    clauses: list[list[tuple[str, bool]]],
    assignment: dict[str, bool],
) -> bool:
    """Each literal is (variable, is_positive)."""
    return all(
        any(assignment[variable] == is_positive for variable, is_positive in clause)
        for clause in clauses
    )


def solve_cnf_bruteforce(
    clauses: list[list[tuple[str, bool]]]
) -> dict[str, bool] | None:
    variables = sorted({variable for clause in clauses for variable, _ in clause})

    for values in product((False, True), repeat=len(variables)):
        assignment = dict(zip(variables, values))
        if verify_cnf(clauses, assignment):
            return assignment
    return None


# (x OR not y) AND (y OR z) AND (not x OR z)
formula = [
    [("x", True), ("y", False)],
    [("y", True), ("z", True)],
    [("x", False), ("z", True)],
]
assignment = solve_cnf_bruteforce(formula)
assert assignment is not None
assert verify_cnf(formula, assignment)
print(assignment)
~~~
</details>

For an assignment already supplied, verification scans the literal occurrences once, taking $O(L)$ time where $L$ is the encoded formula length. Exhaustive solving may try $2^n$ assignments for $n$ variables, taking $O(2^nL)$ time. This contrast illustrates NP membership; the claim that SAT is NP-complete additionally depends on a reduction theorem.

**Practice:** [LeetCode 691 - Stickers to Spell Word](https://leetcode.com/problems/stickers-to-spell-word/) is a constrained covering problem whose general relatives are NP-hard; the small target length permits exact state compression.


### **Canonical NP-Complete Problems and Reduction Chains** {#canonical-np-complete-problems-and-reduction-chains}

NP-complete problems form a network rather than an isolated list. A useful repertoire provides source problems with different structures:

| Problem | Decision question | Typical certificate |
|---|---|---|
| SAT / 3SAT | is there a satisfying truth assignment? | assignment to variables |
| Clique | is there a complete subgraph of size at least $k$? | selected vertices |
| Independent Set | is there a pairwise nonadjacent set of size at least $k$? | selected vertices |
| Vertex Cover | is there a cover of size at most $k$? | selected vertices |
| Hamiltonian Cycle | is there a cycle visiting every vertex once? | ordered cycle |
| Subset Sum | does a subset sum exactly to $T$? | selected item indices |
| Set Cover | can at most $k$ sets cover the universe? | selected sets |
| 3-Coloring | can vertices be colored with three colors? | color per vertex |

Choose a reduction source whose constraints resemble the target. Logical targets often begin from 3SAT; selection problems often begin from Independent Set, Vertex Cover, or Set Cover; routing problems may begin from Hamiltonian Cycle.

Three graph problems are related by especially simple transformations. For a graph $G$ with $n$ vertices:

- $S$ is independent in $G$ iff $V\setminus S$ is a vertex cover in $G$;
- $S$ is independent in $G$ iff $S$ is a clique in the complement graph $\overline{G}$;
- therefore parameter $k$ for Independent Set becomes $n-k$ for Vertex Cover and remains $k$ for Clique.

~~~text
INDEPENDENT-SET-TO-CLIQUE(G, k)
    construct complement graph G_bar
    for each distinct u,v
        add edge (u,v) to G_bar iff (u,v) is not an edge of G
    return (G_bar, k)

INDEPENDENT-SET-TO-VERTEX-COVER(G, k)
    return (G, |V| - k)
~~~

![A representative chain of canonical NP-complete problems and the transformations that connect graph formulations.](assets/np-complete-reduction-chain.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: Independent Set transformations</summary>

~~~python
from itertools import combinations


def normalize(edge: tuple[str, str]) -> frozenset[str]:
    return frozenset(edge)


def independent_set_to_clique(
    vertices: set[str],
    edges: list[tuple[str, str]],
    target_size: int,
) -> tuple[set[str], list[tuple[str, str]], int]:
    original = {normalize(edge) for edge in edges}
    complement = [
        (u, v)
        for u, v in combinations(sorted(vertices), 2)
        if frozenset((u, v)) not in original
    ]
    return vertices.copy(), complement, target_size


def independent_set_to_vertex_cover(
    vertices: set[str],
    edges: list[tuple[str, str]],
    target_size: int,
) -> tuple[set[str], list[tuple[str, str]], int]:
    return vertices.copy(), edges.copy(), len(vertices) - target_size


vertices = {"a", "b", "c", "d"}
edges = [("a", "b"), ("b", "c"), ("c", "d")]
independent = {"a", "c"}

_, complement_edges, clique_target = independent_set_to_clique(
    vertices, edges, len(independent)
)
_, _, cover_target = independent_set_to_vertex_cover(
    vertices, edges, len(independent)
)
assert frozenset(("a", "c")) in {normalize(edge) for edge in complement_edges}
assert clique_target == 2
assert cover_target == 2
print(complement_edges)
~~~
</details>

Complement construction inspects all $\binom{n}{2}=O(n^2)$ possible pairs and uses $O(n^2)$ output space in the worst case. The Vertex Cover transformation reuses the graph and changes only the parameter. These costs must be included in the reduction's polynomial-time proof.

**Practice:** [LeetCode 847 - Shortest Path Visiting All Nodes](https://leetcode.com/problems/shortest-path-visiting-all-nodes/) is not Hamiltonian Cycle, but its small-$n$ bitmask state space develops the exact subset reasoning used around Hamiltonian-style problems.


### **Constructing a Reduction Proof** {#constructing-a-reduction-proof}

A reduction proof is not complete when a diagram "looks equivalent." It has four explicit obligations:

1. **Source:** name a problem already known to be hard under the same reduction type.
2. **Construction:** map every source instance to a precisely defined target instance.
3. **Correctness:** prove both directions of the YES-instance equivalence.
4. **Complexity:** show the construction runs in polynomial time and produces polynomial-size output.

Consider the reduction from Independent Set to Vertex Cover. Given $(G=(V,E),k)$, output $(G,|V|-k)$.

**Forward direction.** Suppose $S$ is an independent set with $|S|\ge k$. No edge has both endpoints in $S$, so every edge has at least one endpoint in $V\setminus S$. Thus $V\setminus S$ is a vertex cover of size at most $|V|-k$.

**Reverse direction.** Suppose $C$ is a vertex cover with $|C|\le |V|-k$. If two vertices in $V\setminus C$ were adjacent, their edge would have neither endpoint in $C$, contradicting that $C$ covers every edge. Hence $V\setminus C$ is independent and has size at least $k$.

The graph can be reused or copied and the new parameter computed in $O(|V|+|E|)$ time. Since Vertex Cover also has the polynomial verifier shown earlier, this reduction can participate in an NP-completeness proof.

~~~text
REDUCTION-PROOF-CHECKLIST(source A, target B)
    state known status of A
    define f on every valid instance of A
    prove: source witness => target witness
    prove: target witness => source witness
    bound time and output size of f by a polynomial
    separately prove B is in NP when claiming NP-completeness
~~~

![A complete reduction proof separates the source, construction, bidirectional correctness, and polynomial-time analysis.](assets/reduction-proof-template.svg){fig-align="center" width="100%"}

<details>
<summary>Python property check: Independent Set to Vertex Cover</summary>

~~~python
from itertools import combinations


def exists_independent_set(
    vertices: list[int], edges: list[tuple[int, int]], size: int
) -> bool:
    edge_set = {frozenset(edge) for edge in edges}
    return any(
        all(frozenset(pair) not in edge_set for pair in combinations(chosen, 2))
        for chosen in combinations(vertices, size)
    )


def exists_vertex_cover(
    vertices: list[int], edges: list[tuple[int, int]], limit: int
) -> bool:
    return any(
        all(u in chosen or v in chosen for u, v in edges)
        for size in range(limit + 1)
        for chosen in map(set, combinations(vertices, size))
    )


def reduction_equivalence_holds(
    vertices: list[int], edges: list[tuple[int, int]], k: int
) -> bool:
    return exists_independent_set(vertices, edges, k) == exists_vertex_cover(
        vertices, edges, len(vertices) - k
    )


vertices = [0, 1, 2, 3, 4]
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]
assert all(reduction_equivalence_holds(vertices, edges, k) for k in range(6))
print([reduction_equivalence_holds(vertices, edges, k) for k in range(6)])
~~~
</details>

The property checker is exponential because it solves both problems exhaustively; it is useful for testing a small implementation but is not the proof. The actual reduction is linear in the graph representation, and the logical argument covers every instance at once.

**Practice:** [LeetCode 886 - Possible Bipartition](https://leetcode.com/problems/possible-bipartition/) is polynomial rather than NP-complete, making it useful practice in checking whether structural restrictions change a general partitioning problem into an easier special case.


### **Approximation Algorithms** {#approximation-algorithms}

An **approximation algorithm** runs in polynomial time and returns a feasible solution whose objective is provably close to optimum on every valid input. The guarantee distinguishes it from a heuristic, which may work well empirically but has no required worst-case quality bound.

For minimum Vertex Cover, a simple 2-approximation builds any **maximal matching** $M$: a set of pairwise endpoint-disjoint edges to which no further edge can be added. Add both endpoints of every edge in $M$ to cover $C$.

The result is feasible. If an uncovered edge remained, neither endpoint would belong to a matched edge, so that edge could be added to $M$, contradicting maximality.

The approximation proof uses a lower bound. Because matched edges share no endpoint, every vertex cover must select at least one distinct endpoint for each edge in $M$. Therefore

$$
OPT\ge |M|.
$$

The algorithm selects exactly two endpoints per matched edge:

$$
|C|=2|M|\le 2OPT.
$$

Maximal and maximum are different. A maximal matching cannot be extended locally; it need not contain the largest possible number of edges. Maximality is nevertheless sufficient for both feasibility and the factor-2 proof.

~~~text
APPROXIMATE-VERTEX-COVER(G=(V,E))
    cover <- empty set
    matched_vertices <- empty set
    for each edge (u,v)
        if u and v are both unmatched
            add u and v to matched_vertices
            add u and v to cover
    return cover
~~~

![A maximal matching supplies a lower bound, while selecting both endpoints produces a cover of at most twice optimum.](assets/vertex-cover-2approx.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: 2-approximate Vertex Cover</summary>

~~~python
def approximate_vertex_cover(
    edges: list[tuple[str, str]],
) -> set[str]:
    matched_vertices: set[str] = set()
    cover: set[str] = set()

    for u, v in edges:
        if u not in matched_vertices and v not in matched_vertices:
            # Add one edge to a maximal matching and choose both endpoints.
            matched_vertices.update((u, v))
            cover.update((u, v))

    return cover


def is_cover(edges: list[tuple[str, str]], chosen: set[str]) -> bool:
    return all(u in chosen or v in chosen for u, v in edges)


edges = [
    ("a", "b"),
    ("b", "c"),
    ("c", "d"),
    ("d", "e"),
    ("e", "a"),
    ("b", "d"),
]
cover = approximate_vertex_cover(edges)
assert is_cover(edges, cover)
print(cover)
~~~
</details>

The implementation scans each edge once and performs expected $O(1)$ hash-set operations, giving expected $O(|E|)$ time and $O(|V|)$ auxiliary space. The guarantee is insensitive to edge order, although different orders may return covers of different sizes.

**Practice:** [LeetCode 1723 - Find Minimum Time to Finish All Jobs](https://leetcode.com/problems/find-minimum-time-to-finish-all-jobs/) uses small constraints for exact search; its general scheduling family motivates why approximation guarantees are important at larger scale.


### **Approximation Ratios and Performance Guarantees** {#approximation-ratios-and-performance-guarantees}

Let $ALG(I)$ be the objective value returned by an algorithm on instance $I$, and let $OPT(I)$ be the optimal value.

For a minimization problem, a $\rho$-approximation satisfies

$$
OPT(I)\le ALG(I)\le \rho\,OPT(I),
\qquad \rho\ge 1.
$$

For a maximization problem, the direction reverses:

$$
\frac{OPT(I)}{\rho}\le ALG(I)\le OPT(I).
$$

Equivalently, $ALG/OPT\le\rho$ for minimization and $OPT/ALG\le\rho$ for maximization when the denominators are positive. The ratio is a worst-case multiplicative guarantee over all legal instances, not an average observed on a benchmark.

Some problems use an **additive guarantee**, such as $|ALG-OPT|\le c$, because a multiplicative ratio is meaningless when $OPT=0$. Randomized approximation algorithms must also state whether the bound holds in expectation or with high probability.

An approximation scheme exposes an accuracy parameter $\varepsilon>0$:

- a **PTAS** returns a $(1+\varepsilon)$ approximation for minimization in polynomial time for each fixed $\varepsilon$;
- an **FPTAS** is polynomial in both input length and $1/\varepsilon$.

The dependence on $1/\varepsilon$ distinguishes the two. A running time such as $n^{1/\varepsilon}$ is a PTAS but not an FPTAS.

~~~text
EVALUATE-MINIMIZATION-APPROXIMATION(instance I)
    approximate <- ALG(I)
    optimum <- exact solver on a small validation instance
    if optimum = 0: use an additive or special-case statement
    ratio <- approximate / optimum
    compare ratio with the proved worst-case bound rho
~~~

![Minimization and maximization ratios use opposite orientations, while PTAS and FPTAS expose an accuracy parameter.](assets/approximation-guarantees.svg){fig-align="center" width="100%"}

<details>
<summary>Python experiment: measure the Vertex Cover approximation ratio</summary>

~~~python
from itertools import combinations
from random import Random


def exact_vertex_cover_size(
    vertices: list[int], edges: list[tuple[int, int]]
) -> int:
    for size in range(len(vertices) + 1):
        for chosen_tuple in combinations(vertices, size):
            chosen = set(chosen_tuple)
            if all(u in chosen or v in chosen for u, v in edges):
                return size
    raise AssertionError


def approximate_cover_size(edges: list[tuple[int, int]]) -> int:
    matched: set[int] = set()
    for u, v in edges:
        if u not in matched and v not in matched:
            matched.update((u, v))
    return len(matched)


rng = Random(11)
observed_ratios = []
vertices = list(range(8))
for _ in range(80):
    edges = [
        pair
        for pair in combinations(vertices, 2)
        if rng.random() < 0.25
    ]
    optimum = exact_vertex_cover_size(vertices, edges)
    approximate = approximate_cover_size(edges)
    ratio = 1.0 if optimum == 0 else approximate / optimum
    observed_ratios.append(ratio)
    assert ratio <= 2.0

print(max(observed_ratios))
~~~
</details>

Exact evaluation is exponential and is used only on small validation graphs. Observing ratios below two does not prove the theorem; the maximal-matching lower-bound argument proves the guarantee for every graph.

**Practice:** [LeetCode 1049 - Last Stone Weight II](https://leetcode.com/problems/last-stone-weight-ii/) is a partition optimization problem solved exactly under bounded sums; general knapsack-style families are central examples for pseudopolynomial algorithms and approximation schemes.


### **Randomized Algorithms and Uncertainty** {#randomized-algorithms-and-uncertainty}

A **randomized algorithm** uses random bits as part of its computation. For a fixed input, different random choices create a distribution over running times, outputs, or both. The probability statement concerns internal randomness; it does not require inputs to be randomly generated.

#### **Las Vegas and Monte Carlo Algorithms** {#las-vegas-and-monte-carlo}

A **Las Vegas algorithm** always returns a correct answer, but its running time is random. Randomized quicksort is the standard example: every pivot sequence still sorts correctly, while random pivots make the expected number of comparisons $O(n\log n)$ against any fixed input order. The worst case remains $O(n^2)$.

A **Monte Carlo algorithm** has a bounded running time but may return an incorrect answer with controlled probability. Freivalds' algorithm checks whether $AB=C$ in $O(n^2)$ time per round instead of multiplying matrices in $O(n^3)$ time. If $AB\ne C$, one random binary vector detects the error with probability at least $1/2$; a correct product is always accepted.

Independent repetition amplifies confidence. If one round has failure probability at most $p$, accepting only after $r$ independent rounds leaves failure probability at most

$$
p^r.
$$

For Freivalds' algorithm, $p\le 1/2$, so $r$ rounds reduce false acceptance to at most $2^{-r}$.

#### **Expected Running-Time Analysis** {#expected-running-time-analysis}

If $\Omega$ is the set of random executions and $T(\omega)$ is the running time under outcome $\omega$, then

$$
\mathbb{E}[T]
=
\sum_{\omega\in\Omega}\Pr[\omega]T(\omega).
$$

Expected time is an average over the algorithm's random choices for the fixed input. Linearity of expectation often simplifies analysis because $\mathbb{E}[\sum_i X_i]=\sum_i\mathbb{E}[X_i]$ even when the indicator variables $X_i$ are dependent.

~~~text
RANDOMIZED-QUICKSORT(A)
    choose a uniformly random pivot
    partition into values below, equal to, and above the pivot
    recursively sort the below and above partitions

FREIVALDS(A, B, C, rounds r)
    repeat r times
        choose random binary vector x
        if A(Bx) != Cx: reject
    accept
~~~

![Las Vegas algorithms randomize running time, Monte Carlo algorithms randomize correctness, and repetition reduces error exponentially.](assets/randomized-algorithms.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementations: randomized quicksort and Freivalds' check</summary>

~~~python
from random import Random


def randomized_quicksort(values: list[int], rng: Random) -> list[int]:
    if len(values) <= 1:
        return values.copy()

    pivot = rng.choice(values)
    lower = [value for value in values if value < pivot]
    equal = [value for value in values if value == pivot]
    upper = [value for value in values if value > pivot]
    return (
        randomized_quicksort(lower, rng)
        + equal
        + randomized_quicksort(upper, rng)
    )


def matrix_vector(matrix: list[list[int]], vector: list[int]) -> list[int]:
    return [
        sum(value * vector[column] for column, value in enumerate(row))
        for row in matrix
    ]


def freivalds(
    a: list[list[int]],
    b: list[list[int]],
    c: list[list[int]],
    rounds: int,
    rng: Random,
) -> bool:
    size = len(a)
    for _ in range(rounds):
        vector = [rng.randrange(2) for _ in range(size)]
        b_vector = matrix_vector(b, vector)
        if matrix_vector(a, b_vector) != matrix_vector(c, vector):
            return False
    return True


rng = Random(5046)
values = [7, 2, 9, 2, 5, 1]
assert randomized_quicksort(values, rng) == sorted(values)

a = [[1, 2], [3, 4]]
b = [[2, 0], [1, 2]]
correct_c = [[4, 4], [10, 8]]
wrong_c = [[4, 4], [10, 9]]
assert freivalds(a, b, correct_c, 8, Random(1)) is True
assert freivalds(a, b, wrong_c, 8, Random(1)) is False
print(randomized_quicksort(values, Random(5046)))
~~~
</details>

Randomized quicksort uses $O(n\log n)$ expected time and $O(n^2)$ worst-case time in this copying implementation. Freivalds takes $O(rn^2)$ time and $O(n)$ auxiliary vector space, with one-sided error at most $2^{-r}$. Always report the error model, number of rounds, and independence assumptions.

**Practice:** [LeetCode 384 - Shuffle an Array](https://leetcode.com/problems/shuffle-an-array/) focuses on generating an unbiased random permutation, a foundational requirement for many randomized algorithms.


### **Exact, Heuristic, and Parameterized Approaches** {#exact-heuristic-and-parameterized-approaches}

Once hardness is established, several algorithmic contracts remain available.

An **exact algorithm** always returns an optimal solution but may use exponential time. Branch and bound, subset dynamic programming, SAT solvers, and integer programming can be highly effective because pruning and modern solver engineering avoid exploring most theoretical candidates on structured instances.

A **heuristic** seeks good solutions quickly without a universal approximation guarantee. Local search, greedy repair, metaheuristics, and domain-specific construction rules belong here. Heuristics should be evaluated empirically against lower bounds, exact solutions on small instances, and representative workloads.

**Parameterized complexity** separates a parameter $k$ from total input size $n$. A problem is fixed-parameter tractable when it has running time

$$
f(k)n^c,
$$

where $c$ is constant independent of $k$. This differs from $n^{f(k)}$, which is polynomial for fixed $k$ but belongs to the broader XP form and may scale poorly even for moderate parameters.

Vertex Cover is FPT with parameter $k$. Pick any uncovered edge $(u,v)$. Every cover must include $u$ or $v$, so branch on those two forced possibilities and decrease the remaining budget. The search tree has depth at most $k$ and branching factor two.

~~~text
VERTEX-COVER-FPT(edges E, budget k)
    if E is empty: return empty cover
    if k = 0: return failure
    choose any edge (u,v)

    result <- recurse after selecting u and deleting incident edges
    if result succeeds: return result union {u}

    result <- recurse after selecting v and deleting incident edges
    if result succeeds: return result union {v}
    return failure
~~~

![Exact, approximation, heuristic, and parameterized approaches offer different guarantees and resource tradeoffs.](assets/exact-heuristic-parameterized.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: parameterized Vertex Cover branching</summary>

~~~python
def vertex_cover_fpt(
    edges: list[tuple[str, str]], budget: int
) -> set[str] | None:
    normalized = frozenset(frozenset(edge) for edge in edges)

    def search(
        remaining: frozenset[frozenset[str]], k: int
    ) -> set[str] | None:
        if not remaining:
            return set()
        if k == 0:
            return None

        u, v = tuple(next(iter(remaining)))
        for chosen in (u, v):
            # Selecting chosen covers and removes every incident edge.
            reduced = frozenset(edge for edge in remaining if chosen not in edge)
            suffix = search(reduced, k - 1)
            if suffix is not None:
                return suffix | {chosen}
        return None

    return search(normalized, budget)


edges = [("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")]
assert vertex_cover_fpt(edges, 1) is None
cover = vertex_cover_fpt(edges, 2)
assert cover is not None and len(cover) <= 2
assert all(u in cover or v in cover for u, v in edges)
print(cover)
~~~
</details>

The search has at most $2^k$ leaves. With an $O(|E|)$ edge-filter operation per node, this direct implementation takes $O(2^k|E|)$ time and $O(k|E|)$ copied-state space along a recursion path. More careful representations and kernelization improve constants and state size while preserving fixed-parameter tractability.

**Practice:** [LeetCode 698 - Partition to K Equal Sum Subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/) rewards exact branching, symmetry breaking, and parameter-aware pruning on constrained instances.


### **Choosing a Practical Response to Hardness** {#choosing-a-practical-response-to-hardness}

A hardness result identifies a boundary, but implementation still requires a contract. Start by formalizing the actual problem rather than its informal label:

1. Is the required output decision, witness, count, or optimum?
2. Must the answer be exact, or is a certified gap acceptable?
3. What are realistic values of $n$, numeric weights, and structural parameters?
4. Do instances belong to an easier special class such as bipartite, interval, planar, bounded-treewidth, or small-universe inputs?
5. Can preprocessing reduce the instance without changing the answer?

Then choose an approach whose guarantee matches the requirement:

| Situation | Starting approach | What to report |
|---|---|---|
| small instances, exact answer required | exhaustive search, subset DP, branch and bound | worst-case limit and solved size |
| small parameter $k$ | FPT branching or kernelization | dependence $f(k)n^c$ and chosen parameter |
| large instances, quality guarantee required | approximation algorithm or relaxation | proven ratio or optimality gap |
| large instances, rapid feasible answer required | heuristic or local search | benchmark quality and variance |
| strong solver available | SAT, CP, or ILP formulation | time limit, bound, and final gap |
| streaming or uncertain environment | randomized or online method | probability and adversarial assumptions |

Hybrid **anytime algorithms** are often the most useful: maintain the best feasible solution as an upper bound for minimization, maintain a relaxation or matching lower bound, and stop when time expires with a measured optimality gap. This communicates more than returning a heuristic value without context.

~~~text
PRACTICAL-HARD-PROBLEM-WORKFLOW(instance I)
    formalize output and required guarantee
    detect polynomial special cases
    estimate n, weights, and useful parameter k
    if exact and small: run exact or parameterized method
    else if guarantee required: run approximation or bounded solver
    else: run heuristic with validation and a time budget
    report solution, lower/upper bound, gap, runtime, and assumptions
~~~

![A practical workflow chooses specialized, exact, parameterized, approximation, heuristic, or hybrid methods from the real problem contract.](assets/hardness-response-workflow.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: a small hybrid Vertex Cover strategy</summary>

~~~python
from itertools import combinations


def hybrid_vertex_cover(
    vertices: list[str],
    edges: list[tuple[str, str]],
    exact_vertex_limit: int = 20,
) -> tuple[str, set[str], int]:
    """Return method, cover, and a certified lower bound."""
    if len(vertices) <= exact_vertex_limit:
        for size in range(len(vertices) + 1):
            for chosen_tuple in combinations(vertices, size):
                chosen = set(chosen_tuple)
                if all(u in chosen or v in chosen for u, v in edges):
                    return "exact enumeration", chosen, size

    # For larger graphs, a maximal matching gives both a lower bound and cover.
    matched: set[str] = set()
    matching_size = 0
    for u, v in edges:
        if u not in matched and v not in matched:
            matched.update((u, v))
            matching_size += 1

    return "2-approximation", matched, matching_size


vertices = ["a", "b", "c", "d", "e"]
edges = [("a", "b"), ("b", "c"), ("c", "d"), ("d", "e"), ("e", "a")]
method, cover, lower_bound = hybrid_vertex_cover(vertices, edges)
assert method == "exact enumeration"
assert len(cover) == lower_bound == 3
assert all(u in cover or v in cover for u, v in edges)

method, cover, lower_bound = hybrid_vertex_cover(
    vertices, edges, exact_vertex_limit=3
)
assert method == "2-approximation"
assert all(u in cover or v in cover for u, v in edges)
assert lower_bound <= 3 <= len(cover)
print(method, cover, lower_bound)
~~~
</details>

The exact branch is exponential but deliberately restricted to a configured small threshold. The large-instance branch is linear in $|E|$ under expected constant-time set operations and returns a lower bound from the matching together with a cover of at most twice that bound. A production hybrid should also enforce wall-clock and memory budgets.

**Practice:** [LeetCode 847 - Shortest Path Visiting All Nodes](https://leetcode.com/problems/shortest-path-visiting-all-nodes/) demonstrates the practical principle that an exponential-in-$n$ bitmask method can be the right exact choice when constraints keep $n$ very small.
